# Week 5 · Day 3 — pandas II: groupby, merge and the cleaning pipeline

Nilufar's Monday files, now in pandas, and the DPD computation merged from `installments.csv` + `payments_dirty.csv`.
Walk-through cells first, then the TODO cells that become HW 5.2.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("..")              # this notebook lives in data/python/, so ".." is the data/ folder
AS_OF = pd.Timestamp("2026-10-01")


def check(value, fingerprints, dp=0):
    """Right or wrong without showing the answer: the rounded value's fingerprint must be one of the expected ones."""
    fp = hashlib.sha256(f"nur-w5:{round(float(value), dp):.{dp}f}".encode()).hexdigest()[:12]
    assert fp in fingerprints, "not yet: this result is not the expected one — check your function and run again"

In [2]:
cust = pd.read_csv(DATA / "excel" / "customers_dirty.csv", parse_dates=["birth_date", "registered_at"])
pay = pd.read_csv(DATA / "excel" / "payments_dirty.csv", parse_dates=["paid_at"])
inst = pd.read_csv(DATA / "excel" / "installments.csv", parse_dates=["due_date"])
cust.shape, pay.shape, inst.shape

((6040, 11), (52875, 8), (92193, 7))

## groupby / agg — `GROUP BY` with named outputs

In [3]:
pay.groupby("region").agg(payments=("payment_id", "size"), total=("amount", "sum"), avg=("amount", "mean")).sort_values("total", ascending=False)

,payments,total,avg
region,,,
Tashkent,20444,102610658010,5.019109e+06
Fergana,6338,34455754032,5.436376e+06
Samarkand,6040,27601576456,4.569797e+06
Andijan,4730,24146158053,5.104896e+06
Namangan,4783,21045029365,4.399964e+06
Bukhara,4090,16162993375,3.951832e+06
Termez,3219,15957579346,4.957310e+06
Nukus,3231,13193421339,4.083386e+06


In [4]:
pay["method"].value_counts()          # ten spellings: the week-2 surprise, again

method
card             17564
payme             8870
cash              8760
click             8754
transfer          8747
Card                57
Click               36
PayMe               30
CASH                30
bank transfer       27
Name: count, dtype: int64

## The three repairs, in order: text → dates → duplicates

In [5]:
pay["method"] = pay["method"].str.strip().str.lower().replace({"bank transfer": "transfer"})
pay["method"].value_counts()

method
card        17621
payme        8900
click        8790
cash         8790
transfer     8774
Name: count, dtype: int64

In [6]:
dup_mask = pay.duplicated(subset=["installment_id", "paid_at", "amount"], keep=False)
print(dup_mask.sum(), "rows in", pay[dup_mask].groupby(["installment_id", "paid_at", "amount"]).ngroups, "groups")
pay[dup_mask].sort_values(["installment_id", "payment_id"]).head(6)

50 rows in 25 groups


,payment_id,loan_id,installment_id,paid_at,amount,method,product,region
109,110,18,174,2025-05-28,663300,click,Buy now pay later,Namangan
52868,52869,18,174,2025-05-28,663300,click,Buy now pay later,Namangan
624,625,108,994,2025-03-23,752500,click,Micro loan,Tashkent
52872,52873,108,994,2025-03-23,752500,click,Micro loan,Tashkent
4451,4452,680,7575,2025-06-23,713100,payme,Micro loan,Andijan
52860,52861,680,7575,2025-06-23,713100,payme,Micro loan,Andijan


In [7]:
pay_clean = pay.sort_values("payment_id").drop_duplicates(subset=["installment_id", "paid_at", "amount"], keep="first")
print(len(pay), "->", len(pay_clean), "rows; removed", pay["amount"].sum() - pay_clean["amount"].sum(), "UZS")

52875 -> 52850 rows; removed 58397800 UZS


## merge — the join, with its two safety belts

In [8]:
paid = (pay_clean[pay_clean["paid_at"] <= AS_OF]
        .groupby("installment_id", as_index=False)
        .agg(paid=("amount", "sum"), first_paid=("paid_at", "min")))
m = inst.merge(paid, on="installment_id", how="left", validate="1:1", indicator=True)
m["_merge"].value_counts()

_merge
both          52826
left_only     39367
right_only        0
Name: count, dtype: int64

In [9]:
m["paid"] = m["paid"].fillna(0)
m["matured"] = m["due_date"] <= AS_OF
m["unpaid"] = m["paid"] < m["total_due"]
print("matured", m["matured"].sum(), "| matured & unpaid", (m["matured"] & m["unpaid"]).sum(), "| overpaid", (m["paid"] > m["total_due"]).sum())

matured 54207 | matured & unpaid 1784 | overpaid 0


## TODO — `clean_customers(df)`
Trim and Title-case the two name columns; `birth_date_invalid` = future or before 1920; `birth_date_clean` = the date or NaT when invalid;
drop duplicates on `phone` + `birth_date` keeping the **earliest** `registered_at`; keep `monthly_income` NaN as NaN. Return the frame sorted by `customer_id`.

In [10]:
def clean_customers(df):
    """Nilufar's customers pipeline: one row per real customer (the check below tells you when the count is right)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [11]:
cc = clean_customers(cust)
check(len(cc), ['5bbb43eee927'])
check(cc["birth_date_invalid"].sum(), ['3d8c079a5199'])
check(cc["monthly_income"].isna().sum(), ['ce063233ec5e'])
assert (cc["customer_id"] <= len(cc)).all()                # each person's earliest registration is the one kept
assert cc["first_name"].str.strip().eq(cc["first_name"]).all()
print("clean_customers OK")

NotImplementedError: TODO: write me, then run the assert cell below

## TODO — `clean_payments(df)`: methods to 5 values, 25 duplicates out, `is_forward_dated` flag (rows kept)

In [12]:
def clean_payments(df):
    """Payments without duplicates, method in five clean values, and an is_forward_dated flag (the checks below tell you when it is right)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [13]:
pc = clean_payments(pd.read_csv(DATA / "excel" / "payments_dirty.csv", parse_dates=["paid_at"]))
check(len(pc), ['09b4ca67aa6e']); check(pc["method"].nunique(), ['fd0f351bf0a5'])
check(pc["amount"].sum(), ['8bc23ec3bcd8'])
check(pc["is_forward_dated"].sum(), ['fedd3e356dc2'])
print("clean_payments OK")

NotImplementedError: TODO: write me, then run the assert cell below

## TODO — `dpd_per_loan(inst, pay_clean, as_of)`
One row per loan with `dpd` (0 when nothing is overdue) and `bucket`. Rule: aggregate clean payments with `paid_at <= as_of` per instalment,
merge onto instalments (left, 1:1), unpaid = paid < total_due, matured = due_date <= as_of, dpd = as_of − min due_date of unpaid matured instalments.

In [14]:
def dpd_per_loan(inst, pay_clean, as_of=AS_OF):
    """DataFrame with columns loan_id, dpd, bucket for every loan in inst."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [15]:
d = dpd_per_loan(inst, pc, AS_OF)
check(len(d), ['2e942804390a'])
check((d["dpd"] > 0).sum(), ['3ef8037facd5'])
b = d["bucket"].value_counts().reindex(["0", "1-30", "31-60", "61-90", "90+"])
check(b["0"], ['0e9bf2c5c458']); check(b["1-30"], ['016b6a8bed7b']); check(b["31-60"], ['22ca36701a22'])
check(b["61-90"], ['3eff0abb19c5']); check(b["90+"], ['d1e5a0a46bd0'])
sql = pd.read_csv(DATA / "excel" / "loans_status.csv")[["loan_id", "dpd"]]
assert (d.merge(sql, on="loan_id", suffixes=("", "_sql")).eval("dpd == dpd_sql")).all()
print(f"dpd_per_loan OK — identical to the SQL column for all {len(d):,} loans")

NameError: name 'pc' is not defined

## HW 5.2 output — write the two clean files next to this notebook (the reviewer re-runs this cell)

In [16]:
# cc.to_csv("customers_clean.csv", index=False)
# pc.to_csv("payments_clean.csv", index=False)
# d.to_csv("dpd_per_loan.csv", index=False)